# Monitor YouTube channels and their newest videos

Take a snapshot of a few YouTube channels: subscribers, video count and total views, plus the views, likes and comments of each channel's newest videos. Run it again later and it lists the new uploads, the videos gaining views fastest and how each channel's totals moved. It can also save the transcripts of new uploads, ready to summarise.

The script is [`monitor.py`](https://github.com/okdataapi/youtube-channel-monitor) (standard library only). It calls the [okdata](https://okdata.online/?src=colab-youtube-channel-monitor) API at $0.002 per call: per channel, one call for its totals, one for its video list and one per video. Three channels with 5 videos each is 21 calls.

## 1. Your API key

Get a key at [okdata.online](https://okdata.online/signup/?src=colab-youtube-channel-monitor) (new keys come with trial credit). In Colab, the easiest way is the key icon in the left sidebar (Secrets): add `OKDATA_API_KEY` and allow this notebook to read it. Otherwise the cell below asks for it. The key is kept in this session only.

In [ ]:
import os, getpass

def find_key():
    try:  # Colab secrets
        from google.colab import userdata
        return userdata.get("OKDATA_API_KEY")
    except Exception:
        pass
    try:  # Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("OKDATA_API_KEY")
    except Exception:
        pass
    return None

if not os.environ.get("OKDATA_API_KEY"):
    os.environ["OKDATA_API_KEY"] = find_key() or getpass.getpass("okdata API key (okd_...): ")
print("key loaded" if os.environ["OKDATA_API_KEY"].startswith("okd_") else "that does not look like an okdata key")

## 2. Get the script

In [ ]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/okdataapi/youtube-channel-monitor/main/monitor.py", "monitor.py")
print("monitor.py downloaded")

## 3. Choose the channels

A handle (`@NASA`), a channel link or a `UC…` channel ID. `--dry-run` shows the plan and the most it can cost before anything is spent.

In [ ]:
import subprocess, sys

CHANNELS = ["@NASA", "@natgeo", "@TED"]
VIDEOS = 5       # newest videos per channel
TRANSCRIPTS = 0  # set to 1 or 2 to also save transcripts of the newest new uploads ($0.004 each)

args = [a for c in CHANNELS for a in ("--channel", c)] + ["--videos", str(VIDEOS), "--transcripts", str(TRANSCRIPTS)]

def run(*extra):
    r = subprocess.run([sys.executable, "monitor.py", *args, *extra], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)

run("--dry-run")

## 4. Take a snapshot

In [ ]:
run()

## 5. Look at the data

In [ ]:
import pandas as pd

channels = pd.read_csv("data/channels.csv")
channels[channels.run_at == channels.run_at.max()][["channel", "name", "subscribers", "videos", "views"]]

In [ ]:
videos = pd.read_csv("data/videos.csv")
latest = videos[videos.run_at == videos.run_at.max()].sort_values("views", ascending=False)
latest[["channel", "title", "published_at", "views", "likes", "comments", "url"]]

In [ ]:
import matplotlib.pyplot as plt

top = latest.head(12).iloc[::-1]
labels = [f"{c} · {str(t)[:34]}" for c, t in zip(top.channel, top.title)]
plt.figure(figsize=(9, 5.5))
plt.barh(labels, top.views)
plt.xlabel("views")
plt.title("Newest videos by views, latest snapshot")
plt.tight_layout()
plt.show()

## 6. Run it again later

Come back in an hour or tomorrow and run the snapshot cell again: the report then lists new uploads, the biggest movers and how each channel's totals changed. A notebook session forgets its files when it ends, so for a lasting daily history fork the [repository](https://github.com/okdataapi/youtube-channel-monitor) and let its GitHub Action take a snapshot every day.

With `TRANSCRIPTS` above 0, the transcripts land in `data/transcripts/<video_id>.txt`, ready to paste into an LLM. The same YouTube data is also available to AI assistants (Claude, Cursor, Dify) through okdata's MCP server: see [okdata.online/mcp](https://okdata.online/mcp/?src=colab-youtube-channel-monitor).